# 16 · Mixing and processing

The flow curves and viscoelastic parameters of the earlier notebooks feed directly into equipment design:
the power a mixer needs, the pressure a die requires, the size of an extrudate after it leaves the die.
This notebook works three such calculations and shows where non-Newtonian behaviour changes the answer.

**What you will learn**
- estimate laminar mixing power for shear-thinning fluids (Metzner-Otto)
- calculate die pressures with a local power law and iteration
- estimate extrudate swell from normal stresses

**Before you start:** notebooks 03, 10 and 15  ·  **Time:** about 60 min

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import flows, models, constitutive as cm
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Mixing a shear-thinning fluid
Shear rates in a stirred tank vary from point to point. Metzner and Otto showed that, in laminar flow, the
power follows the Newtonian curve if the viscosity is evaluated at an average shear rate $k_s N$ (impeller
speed $N$ in rev/s; $k_s \approx$ 10-13 for turbines). For the xanthan drink of notebook 03:

In [2]:
car = (35.0, 0.002, 12.0, 0.25)                             # Carreau fit of the xanthan solution
D_imp, rho, ks, Kp = 0.1, 1000.0, 11.0, 70.0                 # impeller diameter, density, Metzner-Otto constant, Po*Re
for N in (0.5, 1.0, 2.0, 4.0):
    rate = ks * N
    eta = models.viscosity("carreau", rate, *car)
    P = Kp * eta * N**2 * D_imp**3
    Re = rho * N * D_imp**2 / eta
    print(f"N = {N:3.1f} rev/s: average rate {rate:4.0f} 1/s, apparent viscosity {eta:6.3f} Pa s, Re = {Re:6.1f}, power {P*1e3:6.2f} mW")

N = 0.5 rev/s: average rate    6 1/s, apparent viscosity  1.513 Pa s, Re =    3.3, power  26.48 mW
N = 1.0 rev/s: average rate   11 1/s, apparent viscosity  0.901 Pa s, Re =   11.1, power  63.05 mW
N = 2.0 rev/s: average rate   22 1/s, apparent viscosity  0.536 Pa s, Re =   37.3, power 150.18 mW
N = 4.0 rev/s: average rate   44 1/s, apparent viscosity  0.320 Pa s, Re =  125.1, power 358.10 mW


Doubling the speed of a Newtonian mixer in laminar flow quadruples the power; for this shear-thinning drink
the power rises much less, because the fluid thins as it is stirred faster. The check on the Reynolds number
matters: Metzner-Otto holds only in laminar flow, which here ends at the higher speeds.

## Pressure in a sheet die
A polymer melt (the Carreau melt of notebook 02: $\eta_0$ = 3000 Pa·s, $\lambda$ = 0.3 s, $n$ = 0.3) is extruded through a
sheet die 1 m wide with a 1 mm gap and 50 mm land length at 100 kg/h. At process shear rates the melt is
described by a local power law, whose parameters depend on the wall shear rate - so the calculation iterates:

In [3]:
melt = (3000.0, 0.0, 0.3, 0.3)
W, H, L_die, rho_m = 1.0, 1e-3, 0.05, 750.0
Q = 100 / 3600 / rho_m
rate_w = 6 * Q / (W * H**2)                                   # start from the Newtonian wall rate
for it in range(6):
    n_loc = 1 + np.gradient(np.log(models.viscosity("carreau", rate_w * np.array([0.99, 1.0, 1.01]), *melt)), np.log([0.99, 1.0, 1.01]))[1]
    K_loc = models.stress("carreau", rate_w, *melt) / rate_w**n_loc
    res = flows.slit_power_law(K_loc, n_loc, W, H, Q=Q)
    print(f"iteration {it}: local n = {n_loc:.3f}, wall rate {res['rate_w']:7.1f} 1/s, pressure drop {res['dp_per_L']*L_die/1e5:6.2f} bar")
    if abs(res["rate_w"] / rate_w - 1) < 1e-6:
        break
    rate_w = res["rate_w"]

iteration 0: local n = 0.300, wall rate   394.9 1/s, pressure drop  41.89 bar
iteration 1: local n = 0.300, wall rate   395.0 1/s, pressure drop  41.89 bar
iteration 2: local n = 0.300, wall rate   395.0 1/s, pressure drop  41.89 bar


The iteration settles in a few steps. Using the zero-shear viscosity instead would overestimate the
pressure enormously - the melt at the wall is sheared at a rate where its viscosity is far lower.

## Extrudate swell
When an elastic melt leaves a die, the recovering normal stresses make the extrudate swell. Tanner's
estimate uses the ratio of the first normal-stress difference to the shear stress at the wall:

In [4]:
modes = cm.modes_from_spectrum([60.0, 25.0, 8.0], [0.02, 0.2, 2.0], "giesekus", alpha=0.25)   # notebook 10 solution
for rate in (1.0, 10.0, 100.0):
    st = cm.steady_shear(modes, [rate])
    tau = st["eta"][0] * rate
    print(f"wall rate {rate:5.0f} 1/s: N1/tau = {st['N1'][0]/tau:5.2f}, estimated swell ratio {flows.tanner_swell(st['N1'][0], tau):.2f}")

wall rate     1 1/s: N1/tau =  1.63, estimated swell ratio 1.15
wall rate    10 1/s: N1/tau =  3.80, estimated swell ratio 1.29
wall rate   100 1/s: N1/tau =  8.34, estimated swell ratio 1.56


Swell grows with the elasticity of the flow - the ratio $N_1/\tau$, which rises with the Weissenberg number. Die
designers compensate by making the die smaller than the desired product. Tanner's formula is an estimate;
real swell also depends on die length (short dies swell more) and on the full relaxation after exit.

## Inside the algorithm: flow rate through a slit by integrating the profile
For a power-law fluid in a slit the velocity at distance $y$ from the mid-plane is
$u = \frac{n}{n+1}\left(\frac{\Delta P}{KL}\right)^{1/n}\left(h^{1+1/n} - y^{1+1/n}\right)$, $h = H/2$; integrate across the gap:

In [5]:
from scipy import integrate
G = res["dp_per_L"]
u = lambda y: n_loc/(n_loc+1) * (G/K_loc)**(1/n_loc) * ((H/2)**(1+1/n_loc) - y**(1+1/n_loc))
Q_int = 2 * W * integrate.quad(u, 0, H/2, epsabs=0, epsrel=1e-12)[0]
print(f"integrated {Q_int:.6e} m3/s, closed form {res['Q']:.6e} m3/s")

integrated 3.703704e-05 m3/s, closed form 3.703704e-05 m3/s


## Exercises
1. How does the mixing power of the xanthan drink scale with speed between 0.5 and 2 rev/s, compared with a Newtonian liquid? Estimate the exponent from the table.
2. Double the die gap to 2 mm at the same throughput. By what factor does the pressure drop fall, and how does that compare with a Newtonian melt (factor 8)?
3. **Implement it yourself:** write Tanner's swell formula yourself and tabulate the swell ratio against $N_1/\tau$ from 0 to 5. At what ratio does the extrudate swell by 50 %?